In [ ]:
import random
import math
import csv
import json
from collections import OrderedDict, Counter

random.seed(42)

OPERATORS = {
    # binary  (should be ~75-80% of all ops in output)
    "ADD":    {"arity": 2, "weight": 20},
    "SUB":    {"arity": 2, "weight": 15},
    "MUL":    {"arity": 2, "weight": 28},
    "DIV":    {"arity": 2, "weight": 18},
    "POW":    {"arity": 2, "weight": 12},
    # unary — common
    "sin":    {"arity": 1, "weight": 10},
    "cos":    {"arity": 1, "weight": 10},
    "exp":    {"arity": 1, "weight": 8},
    "sqrt":   {"arity": 1, "weight": 7},
    "log":    {"arity": 1, "weight": 5},
    "NEG":    {"arity": 1, "weight": 8},
    # unary — rare
    "tanh":   {"arity": 1, "weight": 2},
    "arcsin": {"arity": 1, "weight": 1},
    "arccos": {"arity": 1, "weight": 1},
    "arctan": {"arity": 1, "weight": 2},
    "tan":    {"arity": 1, "weight": 1},
}

BINARY_OPS = {k: v for k, v in OPERATORS.items() if v["arity"] == 2}
UNARY_OPS  = {k: v for k, v in OPERATORS.items() if v["arity"] == 1}

BINARY_NAMES   = list(BINARY_OPS.keys())
BINARY_WEIGHTS = [BINARY_OPS[k]["weight"] for k in BINARY_NAMES]
UNARY_NAMES    = list(UNARY_OPS.keys())
UNARY_WEIGHTS  = [UNARY_OPS[k]["weight"] for k in UNARY_NAMES]

VARIABLE_POOL  = ["x1", "x2", "x3"]
SPECIAL_CONSTS = {"pi": math.pi, "e": math.e}

NUMERIC_CONSTS = [
    1, 2, 3, 4, 5, 6, 8, 10,
    0.5, 0.25, 0.1, 1.5, 2.5,
    1/3, 2/3, 1/4, 3/4, 1/6,
]
NUMERIC_WEIGHTS = [
    15, 20, 12, 8, 6, 3, 2, 4,
    10, 4, 3, 3, 2,
    3, 2, 2, 2, 1,
]

# ──────────────────────────────────────────────────
# Tree node
# ──────────────────────────────────────────────────
class Node:
    __slots__ = ("kind", "value", "children")
    def __init__(self, kind, value, children=None):
        self.kind = kind
        self.value = value
        self.children = children or []
    def depth(self):
        if not self.children:
            return 1
        return 1 + max(c.depth() for c in self.children)

def pick_binary():
    return random.choices(BINARY_NAMES, weights=BINARY_WEIGHTS, k=1)[0]

def pick_unary():
    return random.choices(UNARY_NAMES, weights=UNARY_WEIGHTS, k=1)[0]

def make_leaf(variables):
    r = random.random()
    if r < 0.50:
        return Node("var", random.choice(variables))
    elif r < 0.85:
        c = random.choices(NUMERIC_CONSTS, weights=NUMERIC_WEIGHTS, k=1)[0]
        return Node("const", c)
    else:
        return Node("special", random.choice(list(SPECIAL_CONSTS.keys())))

# ──────────────────────────────────────────────────
# Tree builder — controls binary/unary ratio
# At each internal node we pick binary vs unary.
# To get ~75% binary ops overall, we need high binary
# probability because binary nodes produce 2 subtrees
# but count as 1 op, while unary produces 1 subtree
# and counts as 1 op. So we need ~92% binary selection
# probability to get ~75% binary in output.
# ──────────────────────────────────────────────────
UNARY_WRAP_PROB = 0.20  # probability of wrapping a subtree with a unary op

def grow_tree(target_depth, variables, current_depth=1):
    if current_depth == target_depth:
        leaf = make_leaf(variables)
        # small chance to wrap leaf in unary (doesn't add depth since we're at target)
        return leaf

    remaining = target_depth - current_depth

    if remaining == 1:
        # exactly 1 level left: place a binary with two leaves, or unary with leaf
        if random.random() < 0.80:
            op = pick_binary()
            return Node("binary", op, [make_leaf(variables), make_leaf(variables)])
        else:
            op = pick_unary()
            return Node("unary", op, [make_leaf(variables)])

    # default: binary op
    op = pick_binary()
    if random.random() < 0.5:
        left  = grow_tree(target_depth, variables, current_depth + 1)
        rd = random.randint(current_depth + 1, target_depth)
        right = grow_tree(rd, variables, current_depth + 1)
    else:
        right = grow_tree(target_depth, variables, current_depth + 1)
        ld = random.randint(current_depth + 1, target_depth)
        left  = grow_tree(ld, variables, current_depth + 1)

    subtree = Node("binary", op, [left, right])

    # optionally wrap the whole subtree in a unary op (adds 1 depth)
    # only do this if we have room (current_depth > 1 means parent handles depth)
    if random.random() < UNARY_WRAP_PROB:
        uop = pick_unary()
        subtree = Node("unary", uop, [subtree])

    return subtree

# ──────────────────────────────────────────────────
# Validation
# ──────────────────────────────────────────────────
BAD_COMBOS = {
    ("NEG", "NEG"), ("exp", "log"), ("log", "exp"),
    ("arcsin", "sin"), ("sin", "arcsin"),
    ("arccos", "cos"), ("cos", "arccos"),
    ("tanh", "tanh"), ("sqrt", "sqrt"),
}

def is_valid(node, ancestors=None):
    if ancestors is None:
        ancestors = set()

    if node.kind == "unary" and node.children:
        c = node.children[0]
        if c.kind == "unary" and (node.value, c.value) in BAD_COMBOS:
            return False

    if node.kind == "unary" and node.value in ancestors and node.value in ("exp", "log", "tan"):
        return False

    if node.kind == "binary" and node.value == "DIV":
        rc = node.children[1]
        if rc.kind == "const" and rc.value == 0:
            return False

    if node.kind == "binary" and node.value == "POW":
        e = node.children[1]
        if e.kind == "const" and e.value not in (0.5, 1, 2, 3, 4, 5, 0.25, 1/3):
            return False
        if e.kind not in ("const", "var", "special"):
            return False

    if node.kind == "unary" and node.value == "exp":
        c = node.children[0]
        if c.kind == "const" and abs(c.value) > 10:
            return False

    new_ancestors = ancestors | ({node.value} if node.kind == "unary" else set())
    for c in node.children:
        if not is_valid(c, new_ancestors):
            return False
    return True

def collect_vars(node):
    if node.kind == "var":
        return {node.value}
    s = set()
    for c in node.children:
        s |= collect_vars(c)
    return s

# ──────────────────────────────────────────────────
# Domain-safe ranges
# Walk tree: any variable that appears ANYWHERE
# inside log/sqrt -> must be positive
# inside arcsin/arccos -> must be bounded [-1,1]
# ──────────────────────────────────────────────────
def collect_var_constraints(node, inside=None):
    """inside tracks what domain-sensitive ops we're nested under."""
    if inside is None:
        inside = set()

    constraints = {}

    if node.kind == "var":
        if "log" in inside or "sqrt" in inside:
            constraints[node.value] = "positive"
        elif "arcsin" in inside or "arccos" in inside:
            constraints[node.value] = "bounded"
        else:
            constraints[node.value] = "any"
        return constraints

    if node.kind in ("const", "special"):
        return constraints

    new_inside = inside.copy()
    if node.kind == "unary" and node.value in ("log", "sqrt", "arcsin", "arccos"):
        new_inside = new_inside | {node.value}

    for c in node.children:
        child_constr = collect_var_constraints(c, new_inside)
        for v, req in child_constr.items():
            old = constraints.get(v, "any")
            constraints[v] = _tightest(old, req)

    return constraints

def _tightest(a, b):
    # "any" < "positive" < "bounded" < "pos_bounded"
    # pos_bounded = needs both positive AND bounded => range (0, 1)
    if {a, b} == {"positive", "bounded"} or "pos_bounded" in (a, b):
        return "pos_bounded"
    rank = {"any": 0, "positive": 1, "bounded": 2, "pos_bounded": 3}
    return a if rank[a] >= rank[b] else b

def make_range(constraint):
    if constraint == "pos_bounded":
        lo = round(random.uniform(0.05, 0.3), 2)
        hi = round(random.uniform(0.5, 0.95), 2)
        return (lo, hi)
    elif constraint == "bounded":
        lo = round(random.uniform(-0.9, -0.1), 2)
        hi = round(random.uniform(0.1, 0.9), 2)
        if lo >= hi:
            lo, hi = -0.5, 0.5
        return (lo, hi)
    elif constraint == "positive":
        lo = round(random.choice([0.1, 0.2, 0.5, 1.0, 0.01]), 2)
        hi = round(lo + random.choice([1.0, 2.0, 3.0, 5.0, 9.0, 10.0]), 2)
        return (lo, hi)
    else:
        lo = round(random.choice([1.0, -1.0, -5.0, -3.0, 0.1, -10.0, 0.5, 2.0]), 2)
        hi = round(lo + random.choice([1.0, 2.0, 3.0, 5.0, 10.0, 6.0]), 2)
        return (lo, hi)

# ──────────────────────────────────────────────────
# Tree -> strings
# ──────────────────────────────────────────────────
def fmt_const(v):
    if isinstance(v, (int, float)) and v == int(v) and abs(v) < 1000:
        return str(int(v))
    for num, den in [(1,3),(2,3),(1,4),(3,4),(1,6)]:
        if abs(v - num/den) < 1e-12:
            return f"{num}/{den}"
    return f"{v:.4g}"

def to_prefix(node):
    if node.kind == "var":     return node.value
    if node.kind == "special": return node.value
    if node.kind == "const":   return fmt_const(node.value)
    if node.kind == "unary":
        return f"{node.value} {to_prefix(node.children[0])}"
    return f"{node.value} {to_prefix(node.children[0])} {to_prefix(node.children[1])}"

def to_infix(node):
    if node.kind in ("var", "special"): return node.value
    if node.kind == "const":            return fmt_const(node.value)
    if node.kind == "unary":
        return f"{node.value} ( {to_infix(node.children[0])} )"
    return f"( {to_infix(node.children[0])} {node.value} {to_infix(node.children[1])} )"

# ──────────────────────────────────────────────────
# Token types & masking
# ──────────────────────────────────────────────────
def get_token_types(prefix_str):
    tokens = prefix_str.split()
    out = []
    for t in tokens:
        if t in BINARY_OPS:     out.append(f"OP_{t}")
        elif t in UNARY_OPS:    out.append("FUNC" if t[0].islower() else f"OP_{t}")
        elif t in VARIABLE_POOL: out.append("VAR")
        elif t in SPECIAL_CONSTS: out.append("CONST")
        else:                    out.append("NUM")
    return " ".join(out)

def mask_constants(prefix_str):
    tokens = prefix_str.split()
    const_dict = OrderedDict()
    counter = 0
    masked = []
    for t in tokens:
        if t in OPERATORS or t in VARIABLE_POOL or t in SPECIAL_CONSTS:
            masked.append(t)
        else:
            try:
                val = eval(t) if "/" in t else float(t)
            except:
                masked.append(t); continue
            existing = None
            for k, v in const_dict.items():
                if abs(v - val) < 1e-12:
                    existing = k; break
            if existing:
                masked.append(existing)
            else:
                key = f"<C_{counter}>"
                const_dict[key] = round(val, 10)
                masked.append(key)
                counter += 1
    return " ".join(masked), dict(const_dict)

# ──────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────
def generate_one(target_depth, num_vars):
    variables = VARIABLE_POOL[:num_vars]
    for _ in range(500):
        tree = grow_tree(target_depth, variables)
        d = tree.depth()
        if d < target_depth or d > target_depth + 1:
            continue
        used = collect_vars(tree)
        if not used:
            continue
        if not is_valid(tree):
            continue

        constraints = collect_var_constraints(tree)
        used_sorted = sorted(used)
        ranges = [make_range(constraints.get(v, "any")) for v in used_sorted]

        prefix = to_prefix(tree)
        infix  = to_infix(tree)
        return {
            "variables": used_sorted,
            "ranges": ranges,
            "expression_infix": infix,
            "expression_prefix": prefix,
            "token_types": get_token_types(prefix),
            "expression_prefix_masked": mask_constants(prefix)[0],
            "constant_dict": mask_constants(prefix)[1],
        }
    return None

def main():
    tasks = [(4, 35000), (6, 10000), (8, 5000)]
    rows = []
    idx = 0

    for depth, count in tasks:
        generated = 0
        seen = set()
        while generated < count:
            nv = random.choices([1, 2, 3], weights=[30, 45, 25], k=1)[0]
            eq = generate_one(depth, nv)
            if eq is None: continue
            if eq["expression_prefix"] in seen: continue
            seen.add(eq["expression_prefix"])

            rows.append({
                "filename": f"EQ_{idx:05d}",
                "variables": json.dumps(eq["variables"]),
                "ranges": json.dumps(eq["ranges"]),
                "expression_infix": eq["expression_infix"],
                "expression_prefix": eq["expression_prefix"],
                "token_types": eq["token_types"],
                "expression_prefix_masked": eq["expression_prefix_masked"],
                "constant_dict": json.dumps(eq["constant_dict"]),
            })
            idx += 1
            generated += 1
            if generated % 10000 == 0:
                print(f"  depth={depth}: {generated}/{count}")

    outpath = "/content/final_last/equations_50k.csv"
    with open(outpath, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nWrote {len(rows)} equations")

    # ── Stats ──
    BINARY = set(BINARY_OPS.keys())
    UNARY  = set(UNARY_OPS.keys())
    op_counts = Counter()
    bc = uc = 0
    for r in rows:
        for tok in r["expression_prefix"].split():
            if tok in BINARY: bc += 1; op_counts[tok] += 1
            elif tok in UNARY: uc += 1; op_counts[tok] += 1
    total = bc + uc
    print(f"\nBinary ops: {bc} ({100*bc/total:.1f}%)")
    print(f"Unary ops:  {uc} ({100*uc/total:.1f}%)")

    print("\n── Operator frequency ──")
    for op, cnt in op_counts.most_common():
        print(f"  {op:8s}: {cnt:7d}  ({100*cnt/total:.1f}%)")

    # Range safety check
    bad = 0
    for r in rows:
        prefix = r["expression_prefix"]
        constraints_needed = set()
        if "log" in prefix.split() or "sqrt" in prefix.split():
            constraints_needed.add("positive")
        ranges = json.loads(r["ranges"])
        for lo, hi in ranges:
            if lo < 0 and "positive" in constraints_needed:
                bad += 1; break
    print(f"\nRange safety — equations with log/sqrt + any negative range: {bad}")

    const_counts = Counter()
    for r in rows:
        cd = json.loads(r["constant_dict"])
        for v in cd.values(): const_counts[v] += 1
    print(f"\n── Top 15 constants ──")
    for c, cnt in const_counts.most_common(15):
        print(f"  {c:>12}: {cnt}")

    print(f"\n── Samples ──")
    for r in rows[:3]:
        for k, v in r.items(): print(f"  {k}: {v}")
        print()

if __name__ == "__main__":
    main()

  depth=4: 10000/35000
  depth=4: 20000/35000
  depth=4: 30000/35000
  depth=6: 10000/10000

Wrote 50000 equations

Binary ops: 331485 (84.4%)
Unary ops:  61431 (15.6%)

── Operator frequency ──
  MUL     :  106932  (27.2%)
  ADD     :   76769  (19.5%)
  DIV     :   68763  (17.5%)
  SUB     :   57651  (14.7%)
  POW     :   21370  (5.4%)
  sin     :   11430  (2.9%)
  cos     :   11249  (2.9%)
  NEG     :    9055  (2.3%)
  exp     :    8430  (2.1%)
  sqrt    :    7900  (2.0%)
  log     :    5438  (1.4%)
  arctan  :    2261  (0.6%)
  tanh    :    2260  (0.6%)
  arcsin  :    1167  (0.3%)
  arccos  :    1127  (0.3%)
  tan     :    1114  (0.3%)

Range safety — equations with log/sqrt + any negative range: 3457

── Top 15 constants ──
           2.0: 19107
           1.0: 15403
           3.0: 12751
           0.5: 11185
           4.0: 9046
           5.0: 7037
          0.25: 7019
          10.0: 4516
  0.3333333333: 3992
           0.1: 3498
           6.0: 3496
           1.5: 3490
  0.66

In [ ]:
import pandas as pd
df = pd.read_csv("/content/final_last/equations_50k.csv")
df.head()

,filename,variables,ranges,expression_infix,expression_prefix,token_types,expression_prefix_masked,constant_dict
0,EQ_00000,"[""x1"", ""x2""]","[[1.0, 7.0], [2.0, 12.0]]",( ( x1 SUB ( x2 POW x1 ) ) ADD ( ( 1 SUB x2 ) ...,ADD SUB x1 POW x2 x1 MUL SUB 1 x2 MUL 1 0.5,OP_ADD OP_SUB VAR OP_POW VAR VAR OP_MUL OP_SUB...,ADD SUB x1 POW x2 x1 MUL SUB <C_0> x2 MUL <C_0...,"{""<C_0>"": 1.0, ""<C_1>"": 0.5}"
1,EQ_00001,"[""x1""]","[[-10.0, -8.0]]",( ( 0.5 SUB cos ( x1 ) ) POW e ),POW SUB 0.5 cos x1 e,OP_POW OP_SUB NUM FUNC VAR CONST,POW SUB <C_0> cos x1 e,"{""<C_0>"": 0.5}"
2,EQ_00002,"[""x1"", ""x2""]","[[-1.0, 5.0], [2.0, 3.0]]",( ( e MUL cos ( 2 ) ) ADD ( ( 4 MUL x2 ) MUL (...,ADD MUL e cos 2 MUL MUL 4 x2 POW e x1,OP_ADD OP_MUL CONST FUNC NUM OP_MUL OP_MUL NUM...,ADD MUL e cos <C_0> MUL MUL <C_1> x2 POW e x1,"{""<C_0>"": 2.0, ""<C_1>"": 4.0}"
3,EQ_00003,"[""x1"", ""x3""]","[[-10.0, -9.0], [-3.0, 0.0]]",sin ( ( ( ( x3 MUL x1 ) MUL ( 1/4 MUL pi ) ) M...,sin MUL MUL MUL x3 x1 MUL 1/4 pi MUL 1 1,FUNC OP_MUL OP_MUL OP_MUL VAR VAR OP_MUL NUM C...,sin MUL MUL MUL x3 x1 MUL <C_0> pi MUL <C_1> <...,"{""<C_0>"": 0.25, ""<C_1>"": 1.0}"
4,EQ_00004,"[""x1""]","[[-0.47, 0.34]]",( ( x1 SUB x1 ) MUL cos ( ( ( x1 SUB 2 ) SUB a...,MUL SUB x1 x1 cos SUB SUB x1 2 arccos x1,OP_MUL OP_SUB VAR VAR FUNC OP_SUB OP_SUB VAR N...,MUL SUB x1 x1 cos SUB SUB x1 <C_0> arccos x1,"{""<C_0>"": 2.0}"


In [ ]:
df.shape

(50000, 8)

In [ ]:
!mv /content/final_last/equations_50k.csv /content/drive/MyDrive/symbolic_data

In [ ]:
"""
Generate 500 data points per equation, save as .npz.

To convert to .pt on your machine:
    python convert_to_pt.py
"""

import csv
import json
import math
import numpy as np
import time
import os

NUM_POINTS = 500
MAX_RESAMPLE = 5
CLIP_VAL = 1e10

OPS = {
    "ADD": lambda a, b: a + b,
    "SUB": lambda a, b: a - b,
    "MUL": lambda a, b: a * b,
    "DIV": lambda a, b: np.where(np.abs(b) < 1e-10, np.nan, a / b),
    "POW": lambda a, b: np.where(
        (a < 0) & (b != np.floor(b)), np.nan,
        np.where(np.abs(b * np.log(np.abs(a) + 1e-30)) > 50, np.nan,
                 np.power(np.abs(a) + 1e-30, b) * np.where(
                     b == np.floor(b), np.sign(a) ** b, 1.0))
    ),
    "sin":    lambda a: np.sin(a),
    "cos":    lambda a: np.cos(a),
    "tan":    lambda a: np.where(np.abs(np.cos(a)) < 1e-10, np.nan, np.tan(a)),
    "exp":    lambda a: np.where(a > 50, np.nan, np.exp(np.clip(a, -50, 50))),
    "sqrt":   lambda a: np.where(a < 0, np.nan, np.sqrt(a)),
    "log":    lambda a: np.where(a <= 0, np.nan, np.log(a)),
    "NEG":    lambda a: -a,
    "tanh":   lambda a: np.tanh(a),
    "arcsin": lambda a: np.where(np.abs(a) > 1, np.nan, np.arcsin(a)),
    "arccos": lambda a: np.where(np.abs(a) > 1, np.nan, np.arccos(a)),
    "arctan": lambda a: np.arctan(a),
}

BINARY_SET = {"ADD", "SUB", "MUL", "DIV", "POW"}
UNARY_SET  = {"sin", "cos", "tan", "exp", "sqrt", "log", "NEG", "tanh", "arcsin", "arccos", "arctan"}
SPECIAL    = {"pi": np.pi, "e": np.e}


def eval_prefix(tokens, variables, X):
    var_map = {v: i for i, v in enumerate(variables)}
    pos = [0]

    def _eval():
        if pos[0] >= len(tokens):
            return np.full(X.shape[0], np.nan)
        tok = tokens[pos[0]]
        pos[0] += 1

        if tok in BINARY_SET:
            left = _eval()
            right = _eval()
            with np.errstate(all="ignore"):
                return OPS[tok](left, right)
        elif tok in UNARY_SET:
            child = _eval()
            with np.errstate(all="ignore"):
                return OPS[tok](child)
        elif tok in var_map:
            return X[:, var_map[tok]]
        elif tok in SPECIAL:
            return np.full(X.shape[0], SPECIAL[tok])
        else:
            try:
                val = eval(tok) if "/" in tok else float(tok)
            except:
                val = np.nan
            return np.full(X.shape[0], val)

    return _eval()


def sample_points(variables, ranges, prefix_str):
    tokens = prefix_str.split()
    num_vars = len(variables)
    all_X, all_y = [], []
    needed = NUM_POINTS

    for _ in range(MAX_RESAMPLE + 1):
        n_sample = min(needed * 3, needed + 2000)
        X = np.empty((n_sample, num_vars), dtype=np.float64)
        for i, (lo, hi) in enumerate(ranges):
            X[:, i] = np.random.uniform(lo, hi, size=n_sample)

        y = eval_prefix(tokens, variables, X)
        valid = np.isfinite(y) & (np.abs(y) < CLIP_VAL)
        all_X.append(X[valid])
        all_y.append(y[valid])

        total = sum(len(a) for a in all_y)
        if total >= NUM_POINTS:
            break
        needed = NUM_POINTS - total

    X_cat = np.concatenate(all_X)[:NUM_POINTS]
    y_cat = np.concatenate(all_y)[:NUM_POINTS]

    if len(y_cat) < NUM_POINTS:
        return None
    return X_cat.astype(np.float32), y_cat.astype(np.float32)


def main():
    csv_path = "/content/drive/MyDrive/symbolic_data/equations_50k.csv"
    out_path = "/content/drive/MyDrive/symbolic_data/final_last_2_march/equations_50k_data.npz"

    print("Loading CSV...")
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    print(f"  {len(rows)} equations")

    np.random.seed(42)
    arrays = {}
    skipped = 0
    t0 = time.time()

    for i, row in enumerate(rows):
        variables = json.loads(row["variables"])
        ranges = json.loads(row["ranges"])
        prefix = row["expression_prefix"]

        result = sample_points(variables, ranges, prefix)

        if result is None:
            skipped += 1
            nv = len(variables)
            arrays[f"X_{i}"] = np.zeros((0, nv), dtype=np.float32)
            arrays[f"y_{i}"] = np.zeros((0,), dtype=np.float32)
        else:
            X_np, y_np = result
            arrays[f"X_{i}"] = X_np
            arrays[f"y_{i}"] = y_np

        if (i + 1) % 5000 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta = (len(rows) - i - 1) / rate
            print(f"  {i+1}/{len(rows)}  ({rate:.0f} eq/s, ETA {eta:.0f}s)  skipped={skipped}")

    elapsed = time.time() - t0
    print(f"\nDone generating in {elapsed:.1f}s")
    print(f"  Valid: {len(rows) - skipped},  Skipped: {skipped}")

    arrays["num_equations"] = np.array(len(rows))

    print(f"\nSaving {out_path}...")
    t1 = time.time()
    np.savez(out_path, **arrays)
    print(f"  Saved in {time.time()-t1:.1f}s")
    print(f"  Size: {os.path.getsize(out_path) / (1024**2):.0f} MB")

    # Verify
    print("\nVerify:")
    d = np.load(out_path)
    n = int(d["num_equations"])
    print(f"  {n} equations in file")
    for j in [0, 1, n//2, n-1]:
        X = d[f"X_{j}"]
        y = d[f"y_{j}"]
        print(f"  [{j}] X:{X.shape} y:{y.shape} y_range=[{y.min():.4f}, {y.max():.4f}]")

    # Write conversion script
    convert_script = '''"""Convert .npz -> .pt (run on your machine with torch installed)"""
import torch, numpy as np

data = np.load("equations_50k_data.npz")
n = int(data["num_equations"])
dataset = []
for i in range(n):
    dataset.append({
        "X": torch.from_numpy(data[f"X_{i}"].copy()),
        "y": torch.from_numpy(data[f"y_{i}"].copy()),
    })
torch.save(dataset, "equations_50k.pt")
print(f"Saved {n} equations to equations_50k.pt")
'''
    # with open("/mnt/user-data/outputs/convert_to_pt.py", "w") as f:
    #     f.write(convert_script)
    # print("Wrote convert_to_pt.py")


if __name__ == "__main__":
    main()

Loading CSV...
  50000 equations
  5000/50000  (2097 eq/s, ETA 21s)  skipped=353
  10000/50000  (1961 eq/s, ETA 20s)  skipped=692
  15000/50000  (2128 eq/s, ETA 16s)  skipped=1023
  20000/50000  (2233 eq/s, ETA 13s)  skipped=1371
  25000/50000  (2317 eq/s, ETA 11s)  skipped=1742
  30000/50000  (2382 eq/s, ETA 8s)  skipped=2088
  35000/50000  (2434 eq/s, ETA 6s)  skipped=2448
  40000/50000  (2026 eq/s, ETA 5s)  skipped=3213
  45000/50000  (1874 eq/s, ETA 3s)  skipped=3999
  50000/50000  (1604 eq/s, ETA 0s)  skipped=5180

Done generating in 31.2s
  Valid: 44820,  Skipped: 5180

Saving /content/drive/MyDrive/symbolic_data/final_last_2_march/equations_50k_data.npz...
  Saved in 50.4s
  Size: 243 MB

Verify:
  50000 equations in file
  [0] X:(500, 2) y:(500,) y_range=[-22673606.0000, -1.9259]
  [1] X:(500, 1) y:(500,) y_range=[0.3071, 3.0107]
  [25000] X:(500, 1) y:(500,) y_range=[2.8639, 41.5992]
  [49999] X:(500, 1) y:(500,) y_range=[-0.1012, 0.1173]


In [ ]:
import numpy as np

data = np.load("/content/drive/MyDrive/symbolic_data/final_last_2_march/equations_50k_data.npz")

lengths = {}

for key in data.files:
    if key.startswith("X_"):
        lengths[key] = len(data[key])

# Check if all are 500
all_500 = all(l == 500 for l in lengths.values())

print("All equations have 500 points:", all_500)
count = 0
# If not, show problematic ones
if not all_500:
    for k, v in lengths.items():
        if v != 500:
            count+=1
            print(f"{k} has {v} points")

print("Total equations checked:", len(lengths), "count = ",count)

All equations have 500 points: False
X_12 has 0 points
X_28 has 0 points
X_36 has 0 points
X_54 has 0 points
X_62 has 0 points
X_70 has 0 points
X_91 has 0 points
X_95 has 0 points
X_135 has 0 points
X_149 has 0 points
X_177 has 0 points
X_194 has 0 points
X_195 has 0 points
X_225 has 0 points
X_234 has 0 points
X_278 has 0 points
X_308 has 0 points
X_319 has 0 points
X_362 has 0 points
X_383 has 0 points
X_391 has 0 points
X_402 has 0 points
X_413 has 0 points
X_431 has 0 points
X_447 has 0 points
X_466 has 0 points
X_481 has 0 points
X_515 has 0 points
X_543 has 0 points
X_545 has 0 points
X_546 has 0 points
X_550 has 0 points
X_571 has 0 points
X_631 has 0 points
X_671 has 0 points
X_685 has 0 points
X_706 has 0 points
X_715 has 0 points
X_722 has 0 points
X_726 has 0 points
X_734 has 0 points
X_736 has 0 points
X_752 has 0 points
X_760 has 0 points
X_761 has 0 points
X_770 has 0 points
X_777 has 0 points
X_790 has 0 points
X_791 has 0 points
X_793 has 0 points
X_795 has 0 points
X_